# Dissertation

**Author**: Mihai Bojescu<br/>
**University**: University "Alexandru Ioan Cuza" of Iași<br/>
**Dissertation coordinator**: Răschip Mădălina<br/>
**Topics**: Machine learning, Diffusion, Audio, NLP<br/>

## Introduction

This document includes the code for my Dissertation paper, with a focus on Music Generation by the means of Diffusion Models.

## Covered topics

This document aims to include the following sections:
- Project setup;
- Project utilities;
- Data loading and preprocessing;
- Model setup;
- Model overfitting sanity check
- Model training;
- Model inference;

## Project setup

This section includes the setup needed to perform the actions in this document.

This section includes prerequisite quirks for AMD GPUs (my machine).

In [ ]:
%env MIOPEN_USER_DB_PATH="$HOME/.cache/miopen"
%env MIOPEN_CUSTOM_CACHE_DIR="$HOME/.cache/miopen"
%env MIOPEN_COMPILE_PARALLEL_LEVEL=4

%env MIOPEN_FIND_MODE=3

%env HSA_ENABLE_SDMA=1
%env ROCR_VISIBLE_DEVICES=0,1

!python --version

!wget -N https://repo.radeon.com/rocm/manylinux/rocm-rel-7.2/torch-2.9.1%2Brocm7.2.0.lw.git7e1940d4-cp313-cp313-linux_x86_64.whl
!wget -N https://repo.radeon.com/rocm/manylinux/rocm-rel-7.2/torchvision-0.24.0%2Brocm7.2.0.gitb919bd0c-cp313-cp313-linux_x86_64.whl
!wget -N https://repo.radeon.com/rocm/manylinux/rocm-rel-7.2/triton-3.5.1%2Brocm7.2.0.gita272dfa8-cp313-cp313-linux_x86_64.whl
!wget -N https://repo.radeon.com/rocm/manylinux/rocm-rel-7.2/torchaudio-2.9.0%2Brocm7.2.0.gite3c6ee2b-cp313-cp313-linux_x86_64.whl

%pip install \
    torch-2.9.1+rocm7.2.0.lw.git7e1940d4-cp313-cp313-linux_x86_64.whl \
    torchvision-0.24.0+rocm7.2.0.gitb919bd0c-cp313-cp313-linux_x86_64.whl \
    torchaudio-2.9.0+rocm7.2.0.gite3c6ee2b-cp313-cp313-linux_x86_64.whl \
    triton-3.5.1+rocm7.2.0.gita272dfa8-cp313-cp313-linux_x86_64.whl


General dependencies

In [ ]:
%pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/rocm7.2
%pip install jupyter ipywidgets torchcodec soundfile numpy matplotlib pandas tqdm

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import multiprocessing as mp
try:
    mp.set_start_method('fork', force=True)
except RuntimeError:
    pass

import numpy as np
import pandas as pd
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as data
import torchvision.transforms.v2 as v2
import torch.amp as amp
import torch.optim as optim
import soundfile as sf
import tqdm.notebook as tqdm
import matplotlib.pyplot as plt
import IPython.display as display
import os
import glob
import math
import dataclasses
import typing as t

%aimport -torch
%aimport -torchaudio
%aimport -torch.nn
%aimport -math
%aimport -dataclasses

Post-dependency quirks for AMD GPUs

In [ ]:
torch._C._jit_set_profiling_executor(False)
torch._C._jit_set_profiling_mode(False)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = False

In [ ]:
@dataclasses.dataclass
class Device:
    index: int
    name: str
    memory: float

is_acceleration_available = torch.cuda.is_available()
devices = [Device(index=i, name=torch.cuda.get_device_name(i), memory=torch.cuda.get_device_properties(i).total_memory) for i in range(torch.cuda.device_count())]
memory_total_bytes = sum([device.memory for device in devices])

print("🖥️ Setup")
print("")
print(f"Is CUDA/ROCm supported: {is_acceleration_available} {"✅" if is_acceleration_available else "❌"}")
print(f"Memory: {memory_total_bytes // 1000 / 1000:.2f} MB")
print(f"Devices:")

for device in devices:
    print(f"- [ID {device.index}] {device.name} ({device.memory // 1000 / 1000:.2f} MB)")

## Project utilities

This section includes utility functions, such as how many timesteps will be used, and DDPM.

In [ ]:
T = 1000

In [ ]:
def linear_beta_schedule(timesteps, start=0.0001, end=0.02):
    return torch.linspace(start, end, timesteps)

def cosine_beta_schedule(timesteps, s=0.008):
    """
    Cosine schedule from https://arxiv.org/abs/2102.09672

    Args:
        timesteps: total diffusion steps (T)
        s: small offset to prevent singularities

    Returns:
        betas: tensor of shape [T]
    """
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)

    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * torch.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]

    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])

    return torch.clamp(betas, 0.0001, 0.9999)

def sigmoid_beta_schedule(timesteps, start=0.0001, end=0.02, tau=1.0):
    """
    Sigmoid schedule for betas.

    Args:
        timesteps: number of diffusion steps (T)
        start: minimum beta
        end: maximum beta
        tau: controls steepness of the sigmoid (higher = flatter)

    Returns:
        betas: tensor of shape [T]
    """
    steps = timesteps + 1
    x = torch.linspace(-6, 6, steps)
    sig = torch.sigmoid(x / tau)

    sig = (sig - sig.min()) / (sig.max() - sig.min())

    betas = sig * (end - start) + start

    return betas[:-1]


def get_index_from_list(vals, t: torch.Tensor, x_shape: list[int], device="cpu"):
    """
    Returns a specific index t of a passed list of values vals
    while considering the batch dimension.
    """
    batch_size = t.shape[0]
    out = vals.gather(-1, t.cpu())
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1))).to(
        device, non_blocking=True
    )


def forward_diffusion_sample(x_0, t, device="cpu"):
    """
    Takes an image and a timestep as input and
    returns the noisy version of it
    """
    sqrt_alphas_cumprod_t = get_index_from_list(
        sqrt_alphas_cumprod, t, x_0.shape, device
    )
    x_0 = x_0.to(device, non_blocking=True)
    sqrt_one_minus_alphas_cumprod_t = get_index_from_list(
        sqrt_one_minus_alphas_cumprod, t, x_0.shape, device
    )
    noise = torch.randn_like(x_0).to(device, non_blocking=True)

    return sqrt_alphas_cumprod_t * x_0 + sqrt_one_minus_alphas_cumprod_t * noise, noise


betas = cosine_beta_schedule(timesteps=T)

alphas = 1.0 - betas
alphas_prev = torch.nn.functional.pad(alphas[:-1], (1, 0), value=1.0)
alphas_cumprod = torch.cumprod(alphas, axis=0)
alphas_cumprod_prev = torch.nn.functional.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
sqrt_alphas_prev = torch.sqrt(alphas_prev)
sqrt_one_minus_alphas_prev = torch.sqrt(1 - alphas_prev)

## Data loading and preprocessing

Data preprocessing on the audio data involve:
1. Dividing the raw clips into 10s ones;
2. Building Mel Spectrograms using them;
3. Saving the results;

Data preprocessing on the text data
1. Building the Byte-Pair-Encoding dictionaries on the input texts;
2. Saving the results;

⚠️ To integrate existing functionality with the CSV data <br>
⚠️ To build the data preprocessing part for text

Parameters can be found below. Best configuration are generally:
$$
\begin{equation}
\begin{split}
n_{fft} & = 2^n, n \in \mathbb{N} \\
length_{hop} & = \frac{n_{fft}}{4} \\
length_{window} & = n_{fft} \\
n_{mels} & = 2^n, \text{preferably 256}
\end{split}
\end{equation}
$$

In [ ]:
segment_seconds=10
sample_rate=24000

n_mels=256
n_fft=4096
hop_length=1024
window_length=4096


image_channels=1
image_height=256
image_width=235

image_pad_height = 64 * math.ceil(image_height / 64) - image_height
image_pad_width = 64 * math.ceil(image_width / 64) - image_width

batch_size=256

cache=True

The Mel spectrogram dataset class can be found below. The class:
1. Retrieves the files;
2. Performs an initial counting to ensure the right-sizing of the dataset length value;
3. Lazy transforms entries;
4. Ensures caching is available to be used;

In [ ]:
if cache:
    os.makedirs("/tmp/dissertation", exist_ok=True)


In [ ]:
@dataclasses.dataclass
class MelSpectrogramSample:
    file: int
    segment: int

class MelSpectrogramDataset(data.Dataset):
    def __init__(
            self,
            path: str,
            segment_seconds: int = 10,
            sample_rate: int = 24000,
            n_mels: int = 256,
            n_fft: int = 2048,
            hop_length: int = 512,
            window_length: int = 2048,
            transform: t.Callable | None = None,
            cache: bool = False,
            cache_path: str = "/tmp/dissertation"
        ):
        self._path: str = path
        self._sample_rate: int = sample_rate
        self._segment_seconds: int = segment_seconds
        self._segment_samples: int = sample_rate * segment_seconds
        self._transform = transform
        self._cache: bool = cache
        self._cache_path: str = cache_path
        self._to_mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            n_mels=n_mels,
            hop_length=hop_length,
            win_length=window_length,
            f_min=0.0,
            f_max=8000.0
        )
        self._to_decibels = torchaudio.transforms.AmplitudeToDB(top_db=120)
        self._files: list[str] = self._build_files_index(path)
        self._sample_index: list[MelSpectrogramSample] = self._build_sample_index(
            self._files,
            self._segment_samples,
            self._cache,
            self._cache_path
        )

    def _build_files_index(self, path: str):
        return [
            *glob.glob(f"{path}/*.wav"),
            *glob.glob(f"{path}/*.mp3"),
            *glob.glob(f"{path}/**/*.wav"),
            *glob.glob(f"{path}/**/*.mp3")
        ]

    def _build_sample_index(self, files: list[str], segment_samples: int, cache: bool, cache_path: str):
        """
        Builds a "sample index": a way to identify to which sample does the X seconds clip belong to.

        Example:     [MelSpectrogramSample(file=0, segment=0), MelSpectrogramSample(file=0, segment=1), MelSpectrogramSample(file=1, segment=0)]
        Explanation: First 2 X seconds segments belong to sample 0. Last one belongs to sample 1.
        """
        result: list[MelSpectrogramSample] = []

        if cache and os.path.isfile(f"{cache_path}/index.csv"):
            df = pd.read_csv(f"{cache_path}/index.csv")

            for (file_index, segment_index) in zip(df["file"].tolist(), df["segment"].tolist()):
                result.append(MelSpectrogramSample(file=file_index, segment=segment_index))
            
            return result

        for file_index, file in enumerate(files):
            data, sample_rate = sf.read(file, dtype='float32', always_2d=True)
            waveform = torch.from_numpy(data.T)

            if sample_rate != self._sample_rate:
                waveform = torchaudio.functional.resample(waveform, sample_rate, self._sample_rate)

            total_samples = waveform.shape[1]

            for segment_index in range(math.ceil(total_samples / segment_samples)):
                result.append(MelSpectrogramSample(file=file_index, segment=segment_index))

        if cache:
            df = pd.DataFrame(
                [(entry.file, entry.segment) for entry in result],
                columns=["file", "segment"]
            )
            df.to_csv(f"{cache_path}/index.csv")

        return result
    
    def __len__(self):
        return len(self._sample_index)
    
    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, int, str]:
        sample = self._sample_index[index]
        file = self._files[sample.file]

        if self._cache:
            cached_path = f"{self._cache_path}/{sample.file}_{sample.segment}.pt"

            if os.path.isfile(cached_path):
                tensor = torch.load(f"{self._cache_path}/{sample.file}_{sample.segment}.pt")
                start = self._segment_samples * sample.segment
                end = self._segment_samples * (sample.segment + 1)

                return (
                    tensor,
                    start,
                    end,
                    file
                )

        data, sr = sf.read(file, dtype='float32', always_2d=True)
        waveform = torch.from_numpy(data.T)

        if sr != self._sample_rate:
            waveform = torchaudio.functional.resample(waveform, sr, self._sample_rate)

        start = self._segment_samples * sample.segment
        end = self._segment_samples * (sample.segment + 1)
        chunk = waveform[:, start:end]

        if chunk.shape[1] < self._segment_samples:
            pad = self._segment_samples - chunk.shape[1]
            chunk = F.pad(chunk, (0, pad))
        
        # NOTE: Mel spectrograms are between [0, +inf]
        #       AmplitudeToDB outputs [max_db - 120, max_db] where max_db = 10 * log10(max_mel_power),
        #       NOT [−120, 0]. With n_fft=4096 the mel power can be large (max_db ≈ 60 dB),
        #       so we must subtract the per-sample max to anchor the range at exactly [−120, 0]
        #       before the linear scaling to [−1, 1].
        tensor = self._to_mel_spectrogram(chunk)
        tensor = self._to_decibels(tensor)
        tensor = tensor - tensor.amax()
        tensor = (tensor + 120) / 120
        tensor = tensor * 2 - 1

        if self._cache:
            cached_path = f"{self._cache_path}/{sample.file}_{sample.segment}.pt"
            torch.save(tensor, cached_path)

        if self._transform is not None:
            tensor = self._transform(tensor)

        return (
            tensor,
            start,
            end,
            file
        )
    
    def is_cache_enabled(self):
        return self._cache
    
dataset = MelSpectrogramDataset(
    path="/home/mihai/Projects/2026/Dissertation/musicnet/train_data",
    segment_seconds=segment_seconds,
    sample_rate=sample_rate,
    n_mels=n_mels,
    n_fft=n_fft,
    hop_length=hop_length,
    window_length=window_length,
    cache=cache,
    # cache_path="/home/mihai/Projects/2026/Dissertation/cache"
)


Data loading section below.

In [ ]:
dataloader = data.DataLoader(
    dataset=dataset,
    batch_size=batch_size,
    drop_last=True,
    shuffle=True,
    pin_memory=True,
    num_workers=8,
    prefetch_factor=4,
)

Cell below ensures the cache is populated. On an AMD Ryzen 7 7700x, this process takes about 10 minutes. Using the cache reduces load times from several minutes to 4 seconds.

In [ ]:
if cache:
    for entry in tqdm.tqdm(dataloader):
        pass

Sampling from the dataset.

In [ ]:
entry = next(iter(dataloader))[0]

_, channels, n_mels, n_frames = entry.shape
times = torch.arange(n_frames) * hop_length / sample_rate
xticks = torch.linspace(0, n_frames, 6)
xtick_labels = torch.linspace(0, times[-1], 6)

fig = plt.figure()
ax = fig.subplots(1, 1)
ax.imshow(entry[0, :, :, :].permute(1, 2, 0), origin="lower", cmap="plasma")
ax.set_xlabel("Time")
ax.set_xticks(xticks.numpy())
ax.set_xticklabels(np.round(xtick_labels.numpy(), 2))
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency bin (Mel)")
ax.set_title("Spectrogram sample from the dataset")
fig.tight_layout()
fig.show()

num_images = 10
fig = plt.figure(figsize=(20, 15))
axs = fig.subplots(1, num_images)
stepsize = int(T / num_images)

for idx in range(0, T, stepsize):
    t = torch.Tensor([idx]).type(torch.int64)
    img_seed, noise = forward_diffusion_sample(entry, t)

    if len(img_seed.shape) == 4:
        img_seed = img_seed[0, :, :, :]

    axs[idx // stepsize].imshow(img_seed.permute(1, 2, 0), origin="lower", cmap="plasma")

fig.show()

# NOTE: Reverse process should be:
#       1. [-1, 1] to [-120, 0]
#       2. [-120, 0] to [1e-12, 1.0]
#       3. Reverse the mel scale to a spectrogram
#       4. Reverse the spectrogram to a waveform
_inv_mel = torchaudio.transforms.InverseMelScale(
    n_stft=n_fft // 2 + 1,
    n_mels=n_mels,
    sample_rate=sample_rate
)
_griffin = torchaudio.transforms.GriffinLim(
    n_fft=n_fft,
    n_iter=128,
    hop_length=hop_length,
    win_length=window_length,
    power=2.0
)

spec_norm = entry[0].detach().cpu()
spec_db = (spec_norm + 1.0) / 2.0 * 120.0 - 120.0
spec_power = torch.pow(10.0, spec_db / 10.0).float()
spec_power = torch.nan_to_num(spec_power, nan=0.0, posinf=1e6, neginf=0.0)
linear_spec = _inv_mel(spec_power)
waveform = _griffin(linear_spec).squeeze().numpy()

sf.write("/tmp/sample.wav", waveform, sample_rate)


print(f"Dataset statistics")
print(f"")
print(f"Spectrogram:")
print(f"    shape: {entry.shape}")
print(f"      min: {entry.min():.4f}")
print(f"      max: {entry.max():.4f}")
print(f"Reversed spectrogram:")
print(f"    shape: {spec_power.shape}")
print(f"      min: {spec_power.min():.4f}")
print(f"      max: {spec_power.max():.4f}")
print(f"Waveform:")
print(f"    shape: {waveform.shape}")
print(f"      min: {waveform.min():.4f}")
print(f"      max: {waveform.max():.4f}")
print(f"")
print(f"Saved /tmp/sample.wav — shape: {waveform.shape}")

display.Audio(filename="/tmp/sample.wav")

## Model setup

This section includes the actual model that will be trained.

In [ ]:
class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, up=False):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)
        if up:
            self.conv1 = nn.Conv2d(2 * in_ch, out_ch, 3, padding=1)
            self.transform = nn.ConvTranspose2d(out_ch, out_ch, 4, 2, 1)
        else:
            self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
            self.transform = nn.Conv2d(out_ch, out_ch, 4, 2, 1)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bnorm1 = nn.GroupNorm(8, out_ch)
        self.bnorm2 = nn.GroupNorm(8, out_ch)
        self.silu = nn.SiLU()

    def forward(
        self,
        x,
        t,
    ):
        h = self.bnorm1(self.silu(self.conv1(x)))
        time_emb = self.silu(self.time_mlp(t.to(x.device)))
        time_emb = time_emb[(...,) + (None,) * 2]
        h = h + time_emb
        h = self.bnorm2(self.silu(self.conv2(h)))

        return self.transform(h)

class AttentionBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj = nn.Conv2d(channels, channels, 1)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W)
        q, k, v = qkv.unbind(1)
        attn = torch.softmax(q.transpose(-1, -2) @ k / C**0.5, dim=-1)
        h = (attn @ v.transpose(-1, -2)).transpose(-1, -2).reshape(B, C, H, W)
        return x + self.proj(h)

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class SimpleUnet(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        image_channels = in_channels
        down_channels = [64, 128, 256, 512, 1024, 2048]
        up_channels = list(reversed(down_channels))
        out_dim = out_channels
        time_emb_dim = 128

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        self.conv0 = nn.Conv2d(image_channels, down_channels[0], 3, padding=1)

        self.downs = nn.ModuleList(
            [
                Block(down_channels[i], down_channels[i + 1], time_emb_dim)
                for i in range(len(down_channels) - 1)
            ]
        )
        
        self.mid_attn = AttentionBlock(down_channels[-1])

        self.ups = nn.ModuleList(
            [
                Block(up_channels[i], up_channels[i + 1], time_emb_dim, up=True)
                for i in range(len(up_channels) - 1)
            ]
        )

        self.output = nn.Conv2d(up_channels[-1], out_dim, 1)

    def forward(self, x, timestep):
        t = self.time_mlp(timestep)
        x = self.conv0(x)
        residual_inputs = []

        for down in self.downs:
            x = down(x, t)
            residual_inputs.append(x)

        x = self.mid_attn(x)

        for up in self.ups:
            residual_x = residual_inputs.pop()
            x = torch.cat((x, residual_x), dim=1)
            x = up(x, t)

        return self.output(x)


model = SimpleUnet(1, 1)
print("📑 Model parameters:", sum(p.numel() for p in model.parameters()))
model

In [ ]:
@torch.no_grad()
def sample_plot_image(device: torch.device):
    plt.axis("off")
    num_images = 5

    img = torch.randn((1, image_channels, image_height, image_width), device=device)
    fig = plt.figure(figsize=(15, 2))
    axs = fig.subplots(1, num_images)
    stepsize = int(T / num_images)

    for i in range(0, T)[::-1]:
        t = torch.full((1,), i, device=device, dtype=torch.long)
        img = sample_timestep_eta(img, t, 0, device)
        img = torch.clamp(img, -1.0, 1.0)

        if i % stepsize == 0:
            result = img.clone()

            if len(img.shape) == 4:
                result = result[0, :, :, :]

            result = result.detach().cpu().permute(1, 2, 0)
            axs[int(i / stepsize)].imshow(result.detach().cpu().permute(1, 2, 0), origin="lower", cmap="plasma")

    display.display(fig)


@torch.no_grad()
def sample_plot_image_seeded(seed: torch.Tensor, device: torch.device):
    plt.axis("off")
    num_images = 5

    img = seed.detach().clone()
    fig = plt.figure(figsize=(15, 2))
    axs = fig.subplots(1, num_images)
    stepsize = int(T / num_images)

    for i in range(0, T)[::-1]:
        t = torch.full((1,), i, device=device, dtype=torch.long)
        img = sample_timestep_eta(img, t, 0, device)
        img = torch.clamp(img, -1.0, 1.0)

        if i % stepsize == 0:
            result = img.clone()

            if len(result.shape) == 4:
                result = result[0, :, :, :]

            result = result.detach().cpu().permute(1, 2, 0)
            axs[int(i / stepsize)].imshow(result, origin="lower", cmap="plasma")

    display.display(fig)


@torch.no_grad()
def sample_timestep_eta(x, t, eta=0.0, device="cpu"):
    noise_pred = model(x, t)

    sqrt_one_minus_alphas_cumprod_t = get_index_from_list(
        sqrt_one_minus_alphas_cumprod, t, x.shape, device
    )
    sqrt_alphas_cumprod_t = get_index_from_list(
        sqrt_alphas_cumprod, t, x.shape, device
    )
    alphas_cumprod_t = get_index_from_list(alphas_cumprod, t, x.shape, device)
    alphas_cumprod_prev_t = get_index_from_list(alphas_cumprod_prev, t, x.shape, device)

    x0_pred = (x - sqrt_one_minus_alphas_cumprod_t * noise_pred) / sqrt_alphas_cumprod_t

    sigma_t = eta * torch.sqrt(
        (1 - alphas_cumprod_prev_t) / (1 - alphas_cumprod_t)
        * (1 - alphas_cumprod_t / alphas_cumprod_prev_t)
    )

    dir_xt = torch.sqrt(1 - alphas_cumprod_prev_t - sigma_t**2) * noise_pred

    noise = torch.randn_like(x) if sigma_t.item() > 0 else 0.0

    x_prev = torch.sqrt(alphas_cumprod_prev_t) * x0_pred + dir_xt + sigma_t * noise

    return x_prev


## Model overfitting sanity check

Trains a **fresh model on a single spectrogram** for a fixed number of steps.
If the pipeline is correct the loss must converge to near-zero, and the
reverse-diffusion pass must recover a spectrogram that looks like the
original — not like pure noise.

Steps performed:
1. Pull one sample from the dataloader;
2. Instantiate a fresh `SimpleUnet` and overfit it on that single sample;
3. Run full reverse diffusion from pure Gaussian noise;
4. Plot original vs reconstructed spectrogram side-by-side;
5. Convert the reconstruction to audio via Griffin-Lim and play it inline.

In [ ]:
overfit_device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

In [ ]:
overfit_model = SimpleUnet(image_channels, image_channels).to(overfit_device)
overfit_optim = optim.AdamW(overfit_model.parameters(), lr=2e-4, weight_decay=1e-2)
overfit_loss_fn = nn.MSELoss()

In [ ]:
_entry = next(iter(dataloader))[0]
overfit_entry = _entry[0:1].to(overfit_device)
overfit_entry = F.pad(overfit_entry, (image_pad_height, image_pad_width))

fig_gt, ax_gt = plt.subplots(figsize=(10, 4))
ax_gt.imshow(
    overfit_entry[0].permute(1, 2, 0).cpu().numpy(),
    origin="lower", cmap="plasma", aspect="auto"
)
ax_gt.set_title("Ground-truth spectrogram (normalised)")
ax_gt.set_xlabel("Time frame")
ax_gt.set_ylabel("Mel bin")
fig_gt.tight_layout()
fig_gt.show()

print(f"Overfitting entry statistics")
print(f"    shape: {overfit_entry.shape}")
print(f"      min: {overfit_entry.min():.4f}")
print(f"      max: {overfit_entry.max():.4f}")
print(f"")


In [ ]:
overfit_steps = 5000
overfit_losses = []

overfit_model.train()
for step in tqdm.tqdm(range(overfit_steps), desc="overfit"):
    t = torch.randint(0, T, (1,), device=overfit_device).long()
    x_noisy, noise = forward_diffusion_sample(overfit_entry, t, overfit_device)

    overfit_optim.zero_grad()
    noise_pred = overfit_model(x_noisy, t)
    loss = overfit_loss_fn(noise_pred, noise)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(overfit_model.parameters(), 1.0)
    overfit_optim.step()

    overfit_losses.append(loss.item())
    if step % 500 == 0:
        print(f"  step {step:4d} | loss = {loss.item():.6f}")

fig_loss, ax_loss = plt.subplots(figsize=(10, 3))
ax_loss.plot(overfit_losses)
ax_loss.set_xlabel("Step")
ax_loss.set_ylabel("MSE loss")
ax_loss.set_title("Overfit loss — must converge to near-zero")
fig_loss.tight_layout()
fig_loss.show()


In [ ]:
overfit_model.eval()
recon = torch.randn((1, image_channels, image_height, image_width), device=overfit_device)
recon = F.pad(recon, (0, 256 - image_width))

with torch.no_grad():
    for i in tqdm.tqdm(range(0, T)[::-1], desc="reverse diffusion"):
        t_r = torch.full((1,), i, device=overfit_device, dtype=torch.long)
        noise_pred_r = overfit_model(recon, t_r)

        sqrt_one_minus_ac_t = get_index_from_list(sqrt_one_minus_alphas_cumprod, t_r, recon.shape, overfit_device)
        sqrt_ac_t = get_index_from_list(sqrt_alphas_cumprod, t_r, recon.shape, overfit_device)
        ac_prev_t = get_index_from_list(alphas_cumprod_prev, t_r, recon.shape, overfit_device)

        x0_pred = (recon - sqrt_one_minus_ac_t * noise_pred_r) / sqrt_ac_t
        x0_pred = torch.clamp(x0_pred, -1.0, 1.0)

        dir_xt = torch.sqrt(torch.clamp(1.0 - ac_prev_t, min=0.0)) * noise_pred_r
        recon = torch.sqrt(ac_prev_t) * x0_pred + dir_xt

recon = torch.clamp(recon, -1.0, 1.0)

fig_cmp, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(
    overfit_entry[0].permute(1, 2, 0).cpu().numpy(),
    origin="lower", cmap="plasma", aspect="auto"
)
axes[0].set_title("Original (ground truth)")
axes[1].imshow(
    recon[0].permute(1, 2, 0).detach().cpu().numpy(),
    origin="lower", cmap="plasma", aspect="auto"
)
axes[1].set_title("Reconstructed (overfit model)")
for ax in axes:
    ax.set_xlabel("Time frame")
    ax.set_ylabel("Mel bin")
fig_cmp.tight_layout()
fig_cmp.show()

spec_norm = recon[0].detach().cpu()
spec_db = (spec_norm + 1.0) / 2.0 * 120.0 - 120.0
spec_power = torch.pow(10.0, spec_db / 10.0).float()
spec_power = torch.nan_to_num(spec_power, nan=0.0, posinf=1e6, neginf=0.0)

print(f"spec_power  stats: min={spec_power.min():.2e}  max={spec_power.max():.4f}")

_inv_mel = torchaudio.transforms.InverseMelScale(
    n_stft=n_fft // 2 + 1,
    n_mels=n_mels,
    sample_rate=sample_rate
)
_griffin = torchaudio.transforms.GriffinLim(
    n_fft=n_fft,
    n_iter=128,
    hop_length=hop_length,
    win_length=window_length,
    power=2.0
)

linear_spec = _inv_mel(spec_power)
waveform_rec = _griffin(linear_spec).squeeze().numpy()

print(f"waveform_rec stats: min={waveform_rec.min():.4e}  max={waveform_rec.max():.4e}  std={waveform_rec.std():.4e}")

peak = np.abs(waveform_rec).max()
if peak > 1e-8:
    waveform_rec = waveform_rec / peak * 0.9
else:
    print("WARNING: waveform peak is essentially zero — reconstruction may be all silence")

sf.write("/tmp/overfit_recon.wav", waveform_rec, sample_rate)
print(f"Saved /tmp/overfit_recon.wav — shape: {waveform_rec.shape}")
display.Audio(filename="/tmp/overfit_recon.wav")

## Model training

This section contains the training part.

In [ ]:
device = torch.device("cuda:0")
model = nn.DataParallel(model, device_ids=[0, 1])
model = model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
loss_function = torch.nn.L1Loss()
# loss_function = torch.nn.MSELoss()
scaler = amp.GradScaler()
epochs = 100

In [ ]:
# torch.save(model.module.state_dict(), "/home/mihai/Projects/2026/Dissertation/models/1.pt")
# model.module.load_state_dict(torch.load("/home/mihai/Projects/2026/Dissertation/models/1.pt", weights_only=True))

In [ ]:
img_seed = torch.randn((1, image_channels, image_height, image_width), device=device)
img_seed = F.pad(img_seed, (0, 256 - image_width))

for epoch in tqdm.tqdm(range(epochs)):
    for step, entry in enumerate(pbar := tqdm.tqdm(dataloader)):
        entry = entry[0].to(device)
        entry = F.pad(entry, (0, 256 - image_width))
        t = torch.randint(0, T, (entry.shape[0],), device=device).long()

        optimizer.zero_grad()

        # with amp.autocast("cuda", enabled=True):
        #     x_noisy, noise = forward_diffusion_sample(entry, t, device)
        #     x_noisy = x_noisy.to(device)
        #     noise = noise.to(device)
        #     noise_pred = model(x_noisy, t)
        #     loss = loss_function(noise_pred, noise)

        # scaler.scale(loss).backward()
        # scaler.unscale_(optimizer)
        # torch.nn.utils.clip_grad_norm_(model.module.parameters(), 2.0)
        # scaler.step(optimizer)
        # scaler.update()

        x_noisy, noise = forward_diffusion_sample(entry, t, device)
        noise_pred = model(x_noisy, t)
        loss = loss_function(noise_pred, noise)
        # torch.nn.utils.clip_grad_norm_(model.module.parameters(), 2.0)
        loss.backward()
        optimizer.step()

        pbar.set_description(f"Loss {loss.item():.8f}")

        if epoch % 5 == 0 and step == 0:
            sample_plot_image_seeded(img_seed, device)

## Model inference

This section includes an inference example

In [ ]:
device = torch.device("cpu")
sample = torch.randn((1, image_channels, image_height, image_width), device=device)
sample = sample.detach().clone()

model.eval()
with torch.no_grad():
    for i in tqdm.tqdm(range(0, T)[::-1]):
        t = torch.full((1,), i, device=device, dtype=torch.long)
        sample = sample_timestep_eta(sample, t, 0, device)

sample = torch.clamp(sample, -1.0, 1.0)
sample = sample.squeeze(1)

fig_inf, ax_inf = plt.subplots(figsize=(10, 4))
ax_inf.imshow(sample.detach().permute(1, 2, 0).cpu().numpy(), origin="lower", cmap="plasma", aspect="auto")
ax_inf.set_title("Inferred spectrogram (untrained model = noise)")
ax_inf.set_xlabel("Time frame")
ax_inf.set_ylabel("Mel bin")
fig_inf.tight_layout()
fig_inf.show()

spec_db = (sample - 1.0) / 2.0 * 80.0
spec_power = torch.pow(10.0, spec_db / 10.0).float()
spec_power = torch.nan_to_num(spec_power, nan=0.0, posinf=1e6, neginf=0.0)

inverse_mel = torchaudio.transforms.InverseMelScale(
    n_stft=n_fft // 2 + 1,
    n_mels=n_mels,
    sample_rate=sample_rate
)
griffin_lim = torchaudio.transforms.GriffinLim(
    n_fft=n_fft,
    n_iter=64,
    hop_length=hop_length,
    win_length=window_length,
    power=2.0
)

linear_spec = inverse_mel(spec_power)
waveform = griffin_lim(linear_spec)
waveform = waveform.squeeze().numpy()

sf.write("/tmp/inference.wav", waveform, sample_rate)
display.Audio(filename="/tmp/inference.wav")